# Phân tích subject shift ở mức input và vai trò của Riemannian alignment (RA)

Notebook này **chỉ phân tích RA**, không liên quan đến DA và TTA. **Không huấn luyện mô hình nào**, **không cần GPU**: runtime CPU là đủ.

**Cách làm (khớp đúng pipeline của bài):** load dữ liệu, chia session và tính whitener RA đều dùng code của nhánh `feature/hada-full-mamba-autodial`. R̄ của mỗi người tính từ **session đầu** (2a: session 1; 2b: session 1–3; Zhou2016: session 1–2), rồi áp lên **session đánh giá** (2a: session 2; 2b: session 4–5; Zhou2016: session 3). Mọi khoảng cách được **chia cho √(số kênh)** để so sánh được giữa 3 dataset.

**Phần 1: RA giảm shift đến đâu (trên session đánh giá, phần RA chưa từng thấy)**
- *To source*: khoảng cách từ session đánh giá của người target đến tâm dữ liệu train của các người source, trước và sau RA.
- Sau RA, khoảng cách này **bằng đúng shift giữa hai session của chính người đó** (đẳng thức toán học). Nghĩa là RA xóa phần **giữa người với người** và không thể chạm tới phần **giữa các ngày ghi**.

**Phần 2: vì sao RA giúp BCI IV-2a nhiều nhất, hai giả thuyết**
- **H1:** RA có lợi chủ yếu nhờ **làm đồng nhất dữ liệu train** của các người source. Đo bằng shift giữa các người trên session train **trước RA** (sau RA bằng 0 do cách xây dựng). *Kỳ vọng:* 2a lớn nhất.
- **H2:** phần shift RA xóa trên 2a chủ yếu là **hình dạng không gian** (các kênh phối hợp khác đi), không phải **độ lớn** (mọi hướng mạnh hay yếu như nhau). Mỗi khoảng cách được tách chính xác: d²/C = scale² + shape². Khác biệt độ lớn để bộ lọc không gian vẫn đúng chỗ; khác biệt hình dạng làm bộ lọc nhắm sai. *Kỳ vọng:* trên 2a, phần hình dạng chiếm gần hết.
- Khác biệt giữa các dataset được kiểm định bằng **Kruskal-Wallis** và **Mann-Whitney từng cặp (hiệu chỉnh Holm)** trên giá trị từng người.

**Kết quả lưu vào Google Drive** (`MyDrive/TCFormer-results/ra_shift/`): `ra_hypotheses` (hình chính cho H1/H2), `ra_distance_bars`, `ra_tsne_bcic2a`, `ra_pairwise_bcic2a` (PNG để xem, PDF cho bài), cùng `ra_shift_summary.json` và `ra_hypothesis_tests.json`.

**Chạy:** *Runtime → Run all*. Dữ liệu đã tải ở lần trước được dùng lại từ Drive.

In [ ]:
# 1) Mount Drive, clone the code branch, and install the paper's environment (CPU only).
import os
import shutil
import subprocess
import sys
from pathlib import Path
from google.colab import drive

CODE_BRANCH = 'feature/hada-full-mamba-autodial'   # loaders, session splits and RA used in the paper
SCRIPT_BRANCH = 'feature/hada-full-mamba'          # analysis/ra_shift_analysis.py
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/content/tcformer-ra-shift')

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
BNCI_DATA = DRIVE / 'datasets/BNCI_MNE'
ZHOU_DATA = DRIVE / 'datasets/Zhou2016_MNE'
OUTPUT_DIR = DRIVE / 'TCFormer-results/ra_shift'
for path in (BNCI_DATA, ZHOU_DATA, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)


def run(command, cwd=None):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)


if (REPO_PATH / '.git').is_dir():
    run(['git', 'fetch', 'origin', CODE_BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', '--force', CODE_BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{CODE_BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', CODE_BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

# The analysis script lives on the paper branch; copy it into the code checkout.
run(['git', 'fetch', 'origin', SCRIPT_BRANCH], cwd=REPO_PATH)
script = subprocess.run(['git', 'show', 'FETCH_HEAD:analysis/ra_shift_analysis.py'],
                        cwd=REPO_PATH, check=True, capture_output=True, text=True).stdout
(REPO_PATH / 'ra_shift_analysis.py').write_text(script)
assert '_riemannian_align_many' in (REPO_PATH / 'datamodules/base.py').read_text()

# Same Python 3.10 environment as the training notebooks, with CPU-only PyTorch.
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1',
     '--index-url', 'https://download.pytorch.org/whl/cpu'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
print('Environment ready.')

In [ ]:
# 2) Run the analysis (downloads the datasets on first use; no model is trained).
env = os.environ.copy()
env.update({
    'PYTHONUNBUFFERED': '1',
    'MPLBACKEND': 'Agg',  # the kernel's inline backend is unknown to the venv
    'MNE_DATA': str(BNCI_DATA),
    'MNE_DATASETS_BNCI_PATH': str(BNCI_DATA),
    'MNE_DATASETS_ZHOU2016_PATH': str(ZHOU_DATA),
})
process = subprocess.Popen([str(PYTHON), 'ra_shift_analysis.py', str(OUTPUT_DIR)],
                           cwd=REPO_PATH, env=env, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
if process.wait() != 0:
    raise RuntimeError('ra_shift_analysis.py failed; see the output above.')

In [ ]:
# 3) Show the results and the figures.
import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

summary = json.loads((OUTPUT_DIR / 'ra_shift_summary.json').read_text())
tests = json.loads((OUTPUT_DIR / 'ra_hypothesis_tests.json').read_text())

display(Markdown('### Phần 1: shift đến source domain trên session đánh giá (d/√C)'))
rows = []
for name, s in summary.items():
    n = np.sqrt(s['n_channels'])
    rows.append({'Dataset': name, 'Channels': s['n_channels'],
                 'To source, before RA': np.mean(s['to_source_before']) / n,
                 'To source, after RA (= session shift)': np.mean(s['to_source_after']) / n,
                 'Reduction (%)': s['to_source_reduction_percent'],
                 'Silhouette by subject, before': s['silhouette_by_subject_before'],
                 'Silhouette by subject, after': s['silhouette_by_subject_after']})
display(pd.DataFrame(rows).set_index('Dataset').round(3))

display(Markdown('### Phần 2: hai giả thuyết (giá trị trên mỗi kênh, trung bình qua các người)'))
rows = []
for name, s in summary.items():
    rows.append({'Dataset': name,
                 'H1: train shift before RA': np.mean(s['h1_train_shift']),
                 'H1: after RA (should be 0)': np.mean(s['h1_train_shift_after_ra']),
                 'H2: train scale': np.mean(s['h2_train_scale']),
                 'H2: train shape': np.mean(s['h2_train_shape']),
                 'H2: shape fraction': np.mean(s['h2_train_shape_fraction']),
                 'Eval: scale removed by RA': np.mean(s['scale_removed_on_eval']),
                 'Eval: shape removed by RA': np.mean(s['shape_removed_on_eval'])})
display(pd.DataFrame(rows).set_index('Dataset').round(3))

rows = []
for key, label in [('h1', 'H1: train shift'), ('h2_shape_fraction', 'H2: shape fraction'),
                   ('h2_train_shape', 'H2: train shape'),
                   ('h2_shape_removed_on_eval', 'Eval: shape removed by RA')]:
    t = tests[key]
    row = {'Measure': label, 'Kruskal-Wallis p': t['kruskal_p']}
    row.update({f'{pair} (p_Holm)': v['p_holm'] for pair, v in t['pairwise'].items()})
    rows.append(row)
display(Markdown('### Kiểm định giữa các dataset'))
display(pd.DataFrame(rows).set_index('Measure').round(4))

for stem, title in [
    ('ra_hypotheses', 'H1 and H2'),
    ('ra_distance_bars', 'Distance of each target evaluation session to the source domain'),
    ('ra_tsne_bcic2a', 'BCI IV-2a evaluation trials, coloured by subject'),
    ('ra_pairwise_bcic2a', 'BCI IV-2a inter-subject distances (evaluation session)'),
]:
    display(Markdown(f'### {title}'))
    display(Image(filename=str(OUTPUT_DIR / f'{stem}.png'), width=950))
print('Files (PNG for viewing, PDF for the paper):', OUTPUT_DIR)

## Cách đọc kết quả

**Phần 1**
- *Reduction (%)* càng lớn: RA càng kéo người mới lại gần dữ liệu đã học.
- *After RA (= session shift)*: phần còn lại sau RA chính là thay đổi giữa hai ngày ghi của người đó. RA về nguyên lý không xử lý được phần này.

**H1, hình (a)**
- Nếu 2a có shift trên session train **lớn nhất** và kiểm định có ý nghĩa (p_Holm < 0.05 khi so 2a với hai dataset kia), thì **H1 được ủng hộ**: các người source trong 2a khác nhau nhiều nhất, nên việc RA đưa họ về cùng một điểm có lợi nhiều nhất.

**H2, hình (b) và (c)**
- (b) Nếu trên 2a phần **shape** chiếm gần hết (*shape fraction* gần 1), còn trên 2b phần **scale** đáng kể, thì **H2 được ủng hộ** ở mức dữ liệu train.
- (c) Lượng scale và shape mà RA xóa trên session đánh giá: cho biết RA sửa loại shift nào trên dữ liệu chưa từng thấy.

**Nếu kết quả không như kỳ vọng** (ví dụ 2b có shift train lớn nhất), thì giả thuyết đó **bị bác bỏ** và phải viết lại cho trung thực trong bài.

Sau khi chạy xong, gửi lại `ra_shift_summary.json`, `ra_hypothesis_tests.json` và hình `ra_hypotheses.png`.